# 20 · Phase 3: Rhetorical Role prediction on top of the Phase-1 models
Each document is one token stream; sentences are pooled from their token range and classified (13 roles).
`context='document'` reads every sentence inside overlapping windows with its neighbours; `context='sentence'` reads
each sentence alone (the sentence-only vs context-aware comparison of §9.1). Model selection on the official dev sets
(CL_dev + IT_dev), reporting **overall and per-domain** macro-F1 + per-label F1. Runs go to `MyDrive/cs781/rr_runs/`.

Brief (§9.1–9.2): **all four** Phase-1 models; ≥4 methodologically different experiments; **LoRMA compulsory**;
**1 knowledge-distillation** run (`kd_teacher=`; teacher and student must use the same context/window/stride).
Your adaptation methods in `adapters/` (LoRMA, LoRA) work here unchanged.

In [ ]:
# ---- paths + locate the code (lm_lib/ and models/) ----
# Put the code on Drive in ONE of these ways:
#   (a) upload the folders lm_lib/ and models/ (+ hf_token.txt) into MyDrive/cs781_capstone/   [or any folder]
#   (b) upload cs781_code.zip (made by make_code_zip.py) anywhere in MyDrive -> it is unzipped automatically
from google.colab import drive
drive.mount('/content/drive')
import sys, os, glob, zipfile, importlib, shutil, time
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # less fragmentation; must precede CUDA init
MYDRIVE    = '/content/drive/MyDrive'
DRIVE_ROOT = f'{MYDRIVE}/cs781'            # data + runs live here
DRIVE_DATA = f'{DRIVE_ROOT}/data'
RUNS_DIR   = f'{DRIVE_ROOT}/runs'
LOCAL_DATA = '/content/data'

def find_code_dir():
    levels = ['', '*/', '*/*/', '*/*/*/']
    # 1) a folder on Drive containing lm_lib/ (newest wins if there are several copies)
    hits = [h for lv in levels for h in glob.glob(f'{MYDRIVE}/{lv}lm_lib/trainer.py')]
    if hits:
        return os.path.dirname(os.path.dirname(max(hits, key=os.path.getmtime)))
    # 2) the newest cs781_code*.zip on Drive -> always re-extracted fresh (so an updated zip replaces old code)
    zips = [z for lv in levels for z in glob.glob(f'{MYDRIVE}/{lv}cs781_code*.zip')]
    if zips:
        z = max(zips, key=os.path.getmtime)
        shutil.rmtree('/content/cs781_code', ignore_errors=True)
        with zipfile.ZipFile(z) as f:
            f.extractall('/content/cs781_code')
        print('unzipped', z, '| zip modified', time.ctime(os.path.getmtime(z)))
        return '/content/cs781_code'
    raise FileNotFoundError('Could not find lm_lib/ on Drive. Upload the lm_lib/ and models/ folders '
                            '(e.g. to MyDrive/cs781_capstone/) or upload cs781_code.zip to MyDrive, then re-run.')

CODE_DIR = find_code_dir()
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
importlib.invalidate_caches()
# if lm_lib/models were imported earlier in this kernel, drop them so the fresh code is used
for _m in [m for m in list(sys.modules) if m == 'lm_lib' or m.startswith(('lm_lib.', 'models.')) or m == 'models']:
    del sys.modules[_m]
print('CODE_DIR =', CODE_DIR, '| contents:', sorted(os.listdir(CODE_DIR)))
os.makedirs(DRIVE_DATA, exist_ok=True); os.makedirs(RUNS_DIR, exist_ok=True)

In [ ]:
!pip -q install ijson "huggingface_hub[hf_xet]" tokenizers
import os
from huggingface_hub import login, whoami

def get_hf_token():
    if os.environ.get('HF_TOKEN'):
        return os.environ['HF_TOKEN'].strip()
    p = f'{CODE_DIR}/hf_token.txt'
    if os.path.exists(p):
        return open(p).read().strip()
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception as e:
        raise RuntimeError(f'No HF token found: set $HF_TOKEN or create {p}') from e

HF_TOKEN = get_hf_token()
login(token=HF_TOKEN, add_to_git_credential=False)
print('logged in to Hugging Face as', whoami(token=HF_TOKEN)['name'])

In [ ]:
from huggingface_hub import hf_hub_download
import shutil
P3 = '/content/p3'; os.makedirs(P3, exist_ok=True); os.makedirs(f'{DRIVE_ROOT}/data/phase3', exist_ok=True)
for f in ['Train/CL_train.json', 'Train/CL_dev.json', 'Train/IT_train.json', 'Train/IT_dev.json', 'Train/label_vocab.json',
          'Test/CL_test.json', 'Test/IT_test.json']:
    p = hf_hub_download('Exploration-Lab/CS-781-Capstone', f'Phase_3_RR/{f}', repo_type='dataset', token=HF_TOKEN)
    shutil.copyfile(p, f'{P3}/{os.path.basename(f)}')
    shutil.copyfile(p, f'{DRIVE_ROOT}/data/phase3/{os.path.basename(f)}')
!ls -lh {P3}

In [ ]:
PHASE1 = {
    'A': f'{RUNS_DIR}/A_qwen_dense_full',
    'B': f'{RUNS_DIR}/B_encdec',
    'C': f'{RUNS_DIR}/C_gdn_hybrid',
    'D': f'{RUNS_DIR}/D_mla_moe',
}
for k, v in PHASE1.items():
    print(k, v, '| ckpts:', sorted(f for f in os.listdir(v) if f.endswith('.pt')) if os.path.isdir(v) else 'MISSING')
if os.path.isdir(PHASE1['C']):
    !pip -q install flash-linear-attention

In [ ]:
# data overview: label distribution per split/domain (macro-F1 weights every role equally -> rare roles matter)
import pandas as pd
from collections import Counter
from lm_lib.rr import data as rd
labels, label2id = rd.load_labels(f'{P3}/label_vocab.json')
rows = {}
for split in ['train', 'dev']:
    for d in rd.load_docs(P3, split):
        rows.setdefault(f"{d['domain']}_{split}", Counter()).update(d['labels'])
pd.DataFrame(rows).reindex(labels).fillna(0).astype(int)

In [ ]:
from lm_lib.rr.trainer import RRConfig, train_rr, predict_test
cfg = RRConfig(
    phase1_run=PHASE1['A'], phase1_ckpt='best',
    method='full',                 # full | linear_probe | partial | lorma | lora | <adapters/NAME.py>
    method_kwargs={},              # {'k': 4} for partial; {'r': 8, 'lorma_alpha': 16} for lorma; ...
    data_dir=P3, out_dir=f'{DRIVE_ROOT}/rr_runs',
    run_name='',                   # default: <phase1 run>__<method>__<context>
    domains=['CL', 'IT'],          # train domains (dev scores are always reported for both)
    context='document',            # document | sentence
    pooling='mean',                # mean | last
    window=1024, stride=768,       # B (encoder trained on 768): consider window=768, stride=512
    max_sent_tokens=256,           # longer sentences are truncated (must be <= window - stride in document mode)
    batch_size=8,                  # sentence context: examples are single sentences -> a larger batch (e.g. 64) is faster
    grad_accum=1,                  # T4 (16 GB): batch_size=2, grad_accum=4 keeps the effective batch at 8
    grad_checkpoint=False,
    stop_at='',                    # e.g. '2026-10-16T08:30+05:30': stop there (final dev eval, best.pt kept), then the next cell runs
    epochs=10, lr=5e-5, head_lr=1e-3, weight_decay=0.01, warmup_frac=0.1, dropout=0.1,
    class_weighting='none',        # none | inv_sqrt | inv
    train_on_dev=False,            # True = final model on train+dev (no model selection)
    # LLM-generated (Claude) synthetic mini-judgments weighted to rare roles (synthetic/make_rr_synthetic.py) -- declare in report
    extra_train=[],                # e.g. [f'{CODE_DIR}/synthetic/{n}' for n in ('CL_synth.json', 'IT_synth.json', 'CL_gen.json', 'IT_gen.json')]
    # augmentation from the official training docs only (segment swapping; see lm_lib/rr/augment.py)
    augment_docs=0,                # synthetic docs per domain (e.g. 40); 0 = off
    augment_rare_boost=3.0,        # how much more often templates containing rare roles are used
    oversample_rare=0,             # extra copies of windows that contain a rare-role sentence
    rare_threshold=500,            # roles with fewer training sentences than this are 'rare'
    kd_teacher='',                 # KD: a trained teacher run dir in rr_runs/ (+ kd_alpha, kd_temperature)
)
print(cfg)

In [ ]:
results = train_rr(cfg)

In [ ]:
# CL_test + IT_test -> prediction.json {doc_id: [one label per sentence]} + submission.zip (length checks included)
run_dir = f'{DRIVE_ROOT}/rr_runs/{cfg.run_name}'
zip_path, test_preds = predict_test(run_dir)
from google.colab import files
files.download(zip_path)

In [ ]:
# all Phase-3 runs side by side (for the report)
import json, glob
rows = [json.load(open(p)) for p in glob.glob(f'{DRIVE_ROOT}/rr_runs/*/results.json')]
df = pd.DataFrame(rows)[['run', 'phase1_run', 'method', 'context', 'pooling', 'kd_teacher', 'best_dev_macro_f1',
                         'dev_CL_macro_f1', 'dev_IT_macro_f1', 'trainable_params', 'trainable_pct']]
df.sort_values('best_dev_macro_f1', ascending=False)